# Lazy composition with Dask

The grammar should not force eager evaluation. This vignette constructs a
Dask-backed xarray cube, applies ordinary CubeDynamics verbs, and verifies that
the result remains lazy until an explicit `compute()` boundary.

In [ ]:
import dask.array as da
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v

rng = np.random.default_rng(7)
values = rng.normal(size=(24, 8, 10)).astype("float32")
lazy_values = da.from_array(values, chunks=(6, 4, 5))

cube = xr.DataArray(
    lazy_values,
    dims=("time", "y", "x"),
    coords={
        "time": pd.date_range("2023-01-01", periods=24, freq="MS"),
        "y": np.arange(8),
        "x": np.arange(10),
    },
    name="signal",
    attrs={"source": "deterministic synthetic vignette"},
)

assert cube.chunks is not None
cube

## Build a lazy graph

In [ ]:
result = (
    pipe(cube)
    | v.anomaly(dim="time")
    | v.variance(dim="time", keep_dim=False)
).unwrap()

assert result.dims == ("y", "x")
assert result.chunks is not None
graph_tasks = len(result.data.__dask_graph__())
print(f"Lazy result with {graph_tasks} graph tasks and chunks {result.chunks}")

No values have been materialized by the pipe. Compute only the small final
product when the workflow reaches an intentional execution boundary.

In [ ]:
materialized = result.compute()
assert materialized.chunks is None
assert materialized.shape == (8, 10)
assert np.isfinite(materialized.values).all()
materialized